> 📅 __Date: 2026-09-08__

# 🧩 **Embedding Models & Text Splitters**

> **Goal:** Understand what embedding models are, how to select an embedding model, how OpenAI and open-source embedding models are used, and how different LangChain text splitters divide documents into useful chunks for RAG systems.

> **Prerequisite:** This chapter assumes familiarity with the previous RAG Architecture, RAG Implementation, and Retriever notes.

> **Dependencies:**

```python
!pip install -U langchain
!pip install -U langchain-openai
!pip install -U langchain-huggingface
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-experimental
!pip install -U sentence-transformers
!pip install -U pypdf
!pip install -U tiktoken
```

### **Install All Dependencies**

```python
!pip install -U langchain langchain-openai langchain-huggingface langchain-community langchain-text-splitters langchain-experimental sentence-transformers pypdf tiktoken
```

---

# 🧠 **What is an Embedding?**

> **Embedding = A numerical vector representation of text that captures useful semantic information about that text.**

Instead of representing text only as words or characters, an embedding model converts text into a vector of numbers.

```text
Text
 ↓
Embedding Model
 ↓
Numerical Vector
```

**For example, conceptually:**

```text
"The cat is sleeping"
        ↓
[0.12, -0.43, 0.81, 0.05, ...]
```

The vector itself is not meant to be human-readable. Its purpose is to make text mathematically comparable.

---

# 🔎 **Why Do We Need Embeddings?**

Embeddings are widely used in semantic-search and RAG systems.

```text
User Query
    ↓
Embedding Model
    ↓
Query Vector
    ↓
Compare with Document Vectors
    ↓
Most Relevant Documents
```

This allows semantically related text to be retrieved even when the query and document do not use exactly the same words.

### **Example**

```text
Query:
"How does the Transformer understand word order?"

Document:
"The model injects information about the relative or absolute position of tokens..."
```

The wording is different, but the semantic meaning is related.

> **Key Idea:** Embeddings help convert language into a representation that can be compared using vector similarity.

---

# 🏗️ **Embeddings in RAG**

Embeddings are used in both the **indexing** and **retrieval** stages.

```text
                    INDEXING

Documents
   ↓
Chunking
   ↓
Embedding Model
   ↓
Document Vectors
   ↓
Vector Database
```

**At query time:**

```text
                    RETRIEVAL

User Query
   ↓
Embedding Model
   ↓
Query Vector
   ↓
Vector Database
   ↓
Similarity Search
   ↓
Relevant Chunks
```

> **Important:** The same embedding space should generally be used for document and query embeddings so their vectors are meaningfully comparable.

---

# 🏆 **Embedding Model Leaderboard**

A useful benchmark for comparing embedding models is the **MTEB (Massive Text Embedding Benchmark)**.

🔗 https://huggingface.co/spaces/mteb/leaderboard

The leaderboard can help compare models across different embedding tasks and datasets.

---

# 🎯 **Factors to Consider While Selecting an Embedding Model**

```text
Capability
Security
Cost
Latency
Maximum Input Tokens
Embedding Dimension
Language Support
Domain Performance
```

### **1. Capability**

Consider whether the model performs well for your use case:

```text
Semantic Search
RAG
Clustering
Classification
Multilingual Retrieval
Domain-Specific Retrieval
```

### **2. Security**

Consider where your data is processed.

```text
Hosted API
→ Data is sent to an external service

Self-Hosted Model
→ Model can run inside your own infrastructure
```

For sensitive data, deployment and data-handling requirements become especially important.

### **3. Cost**

Hosted embedding APIs commonly charge based on usage, while self-hosted models have infrastructure and operational costs.

```text
API Cost
→ Pay for usage

Self-Hosted Cost
→ GPU / CPU + Memory + Infrastructure
```

### **4. Latency**

Embedding generation should be fast enough for your workload.

```text
Higher Model Size
        ↓
Potentially Higher Computation
        ↓
Potentially Higher Latency
```

### **5. Maximum Input Tokens**

The embedding model has a maximum input size. Long text may need to be split before embedding.

```text
Long Document
    ↓
Text Splitting
    ↓
Smaller Chunks
    ↓
Embedding Model
```

### **6. Embedding Dimension**

Different models produce vectors of different dimensions.

**For example, one model may produce:**

```text
384-dimensional vector
```

**while another may produce:**

```text
1536-dimensional vector
```

Larger dimensions do not automatically mean better retrieval. Evaluate the model for the actual task.

---

# 🏢 **OpenAI Embedding Models**

**Common OpenAI embedding model names include:**

```text
text-embedding-3-large
text-embedding-3-small
text-embedding-ada-002
```

> **Note:** `text-embedding-ada-002` is an older OpenAI embedding model. Newer projects generally use the `text-embedding-3-*` family.

---

# 🧪 **OpenAI Embeddings with LangChain**

In [1]:
import os
from google.colab import userdata

openai = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = openai

In [2]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

### **Create an Embedding for a Query**

In [3]:
vector = embedding_model.embed_query("Hello")

print(vector[:10])
print(len(vector))

[0.019256591796875, -0.06451416015625, -0.0016851425170898438, 0.07818603515625, 0.021636962890625, -0.0155181884765625, -0.0150299072265625, 0.045745849609375, -0.00591278076171875, -0.045257568359375]
1536


**`embed_query()`** returns the embedding vector for a single query string.

### **Check the Embedding Dimension**

In [4]:
len(embedding_model.embed_query("Hello"))

1536

> **Important:** The vector length depends on the embedding model and its configuration. Do not assume that every OpenAI embedding model produces the same dimension.

---

# 📦 **Embed Multiple Documents**

Embedding models can also embed multiple pieces of text.

In [5]:
texts = [
    "Transformer models use attention.",
    "Positional encoding provides information about token position.",
    "RAG retrieves external context before generation."
]

vectors = embedding_model.embed_documents(texts)

print(len(vectors))
print(len(vectors[0]))

3
1536


### **Difference**

```text
embed_query()
→ Embed one query

embed_documents()
→ Embed multiple document texts
```

---

# 🤗 **Open-Source Embedding Models**

Hugging Face provides many open-source / open-weight embedding models.

🔗 https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=downloads

A common library for using sentence-transformer based embedding models is **`sentence-transformers`**.

---

# 🧪 **Hugging Face Embeddings with LangChain**

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

### **Create an Embedding**

In [7]:
vector = embedding_model.embed_query("Hello")

print(vector[:10])
print(len(vector))

[-0.06277173012495041, 0.054958775639534, 0.05216483771800995, 0.08578997850418091, -0.08274892717599869, -0.07457300275564194, 0.06855469197034836, 0.01839640364050865, -0.08201134204864502, -0.0373847559094429]
384


> **Key Idea:** With a local Hugging Face embedding model, text can be embedded locally without sending the text to a hosted embedding API.

---

# 🆚 **Hosted vs Open-Source Embeddings**

| Feature | Hosted Embedding API | Open-Source / Open-Weight Model |
|---|---|---|
| **Setup** | Usually simple | More setup required |
| **Infrastructure** | Provider manages it | You manage it |
| **Privacy Control** | Depends on provider | Greater control when self-hosted |
| **Latency** | Network-dependent | Can be low locally after setup |
| **Cost Model** | Usage-based in many cases | Infrastructure cost |
| **Customization** | Provider-dependent | More deployment/customization freedom |

---

# 🧠 **Embedding Model Mental Model**

```text
Text
 ↓
Embedding Model
 ↓
Vector
 ↓
Vector Database
 ↓
Similarity Search
 ↓
Relevant Text
```

---

# 🗺️ **From Raw Documents to RAG**

**A typical RAG pipeline can be viewed as:**

```text
AI Model
   ↓
Embedding Model
   ↓
Document Loader
   ↓
Text Splitting / Chunking
   ↓
Vector Database
   ↓
Retriever
   ↓
LLM
```

**More accurately, the indexing path is:**

```text
Document
   ↓
Document Loader
   ↓
Text Splitter
   ↓
Chunks
   ↓
Embedding Model
   ↓
Vector Database
```

**And the runtime path is:**

```text
User Query
   ↓
Embedding Model
   ↓
Retriever
   ↓
Relevant Chunks
   ↓
LLM
   ↓
Answer
```

---

# ✂️ **Text Splitters / Chunking**

Large documents usually need to be divided into smaller pieces before embedding and retrieval.

> **Chunking = Dividing a large document into smaller text segments called chunks.**

### **Why Do We Need Chunking?**

```text
Large Document
      ↓
Too much text for one operation
      ↓
Split into smaller chunks
      ↓
Embed / Index chunks
      ↓
Retrieve only relevant chunks
```

**Chunking helps with:**

```text
Context limits
Retrieval precision
Embedding limits
Latency
Cost
Noise reduction
```

---

# 📄 **Load the PDF**

For the examples in this chapter, we use the Transformer paper.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "/content/NIPS-2017-attention-is-all-you-need-Paper.pdf"
)

docs = loader.load()

/tmp/ipykernel_3975/3330132712.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


The loader returns LangChain `Document` objects containing content and metadata.

```text
Document
├── page_content
└── metadata
```

---

# 📝 **Sample Text**

We can experiment with a small text string before applying splitters to a PDF.

In [9]:
text = """
vjas ajsb kabsc

ajsvcjva javsxj ajbsck

ajsvxaxa xjavc a

acac
"""

---

# 1️⃣ **CharacterTextSplitter**

> **CharacterTextSplitter = A text splitter that uses a specified separator and then combines pieces into chunks according to the configured chunk size and overlap.**

In [10]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=10,
    chunk_overlap=0
)

### **Split Raw Text**

In [11]:
chunks = text_splitter.split_text(text)

print(chunks)

['vjas ajsb kabsc', 'ajsvcjva javsxj ajbsck', 'ajsvxaxa xjavc a', 'acac']


This returns a list of plain strings.

### **Create Document Objects**

In [12]:
documents = text_splitter.create_documents([text])

print(documents)

[Document(metadata={}, page_content='vjas ajsb kabsc'), Document(metadata={}, page_content='ajsvcjva javsxj ajbsck'), Document(metadata={}, page_content='ajsvxaxa xjavc a'), Document(metadata={}, page_content='acac')]


This returns LangChain **`Document`** objects.

### **Split Existing Documents**

In [13]:
chunks = text_splitter.split_documents(docs)

This takes existing **`Document`** objects and returns new chunked **`Document`** objects.

---

# 🧠 **CharacterTextSplitter Mental Model**

**Conceptually:**

```text
Input Text
   ↓
Split using separator
   ↓
Small text pieces
   ↓
Merge pieces until chunk size is reached
   ↓
Final Chunks
```

> **Important: `chunk_size`** is a target/maximum-style constraint used while constructing chunks; the exact behavior also depends on the separator and the text content.

---

# 🔄 **Three Important Methods**

```text
split_text(text)
→ Split plain text
→ Returns strings
```

```text
create_documents([text])
→ Split plain text
→ Returns Document objects
```

```text
split_documents(docs)
→ Split existing Document objects
→ Returns Document objects
```

### **Memory Trick**

```text
TEXT
→ split_text()

TEXT → DOCUMENT
→ create_documents()

DOCUMENT
→ split_documents()
```

---

# 2️⃣ **RecursiveCharacterTextSplitter**

> **RecursiveCharacterTextSplitter = A splitter that tries multiple separators in order and recursively splits chunks that are still larger than the configured chunk size.**

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "],
    chunk_size=100,
    chunk_overlap=20
)

### **How It Works**

The splitter attempts separators from the list in order.

```text
1. "\n\n"
       ↓
If chunk is still too large
       ↓
2. "\n"
       ↓
If chunk is still too large
       ↓
3. " "
```

**Conceptually:**

```text
Large Text
    ↓
Try paragraph boundary
    ↓
Still too large?
    ↓
Try newline
    ↓
Still too large?
    ↓
Try space
    ↓
Smaller chunks
```

### **Example**

In [15]:
chunks = text_splitter.split_text(text)

for i, chunk in enumerate(chunks, start=1):
    print(f"--- Chunk {i} ---")
    print(chunk)

--- Chunk 1 ---
vjas ajsb kabsc

ajsvcjva javsxj ajbsck

ajsvxaxa xjavc a

acac


### **Why Recursive Splitting is Useful**

It tries to preserve larger natural text boundaries before falling back to smaller ones.

```text
Paragraph
   ↓
Sentence / Line
   ↓
Word / Space
```

> **Key Idea:** Recursive splitting is usually a strong general-purpose default for ordinary prose because it attempts to keep larger semantic units together before using smaller separators.

---

# 🆚 **Character vs Recursive Character Splitter**

| Feature | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|---|---|---|
| **Separators** | Mainly one specified separator | Multiple separators |
| **Fallback strategy** | Limited | Recursive |
| **Natural boundaries** | Depends on chosen separator | Better preservation of larger boundaries |
| **General-purpose use** | Simple cases | Common default for general text |
| **Configuration** | Simple | More flexible |

### **Memory Trick**

```text
Character
→ One main separator

Recursive Character
→ Many separators → fallback recursively
```

---

# 3️⃣ **Split on Tokens**

Character counts are not the same as token counts.

> **Token-based splitting = Splitting text using a token-aware length measurement rather than raw character count.**

**LangChain provides a Tiktoken-based constructor:**

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    separators=["\n\n", "\n", " "],
    chunk_size=100,
    chunk_overlap=20
)

Here, **`chunk_size`** and **`chunk_overlap`** are interpreted using the tokenizer's token count.

### **Conceptual Flow**

```text
Text
 ↓
Tokenizer
 ↓
Count Tokens
 ↓
Split into Target Token Size
```

### **Why Use Token-Based Splitting?**

LLMs and many model APIs operate with token limits rather than raw character counts.

```text
Character Size
→ Convenient approximation

Token Size
→ Closer to model context constraints
```

### **Token-Based Mental Model**

```text
Character Splitter
→ How many characters?

Token Splitter
→ How many tokens?
```

---

# 🌐 **Document-Specific Splitters**

Some documents have a known structure, so a generic character splitter may not be ideal.

> **Document-specific splitter = A splitter designed around the structure of a particular document format.**

**Examples include splitters for:**

```text
HTML
Markdown
Python
JavaScript
LaTeX
JSON
Code
```

**LangChain provides many specialized splitters:**

🔗 https://docs.langchain.com/oss/python/integrations/splitters

---

# 🌳 **HTMLHeaderTextSplitter**

An HTML document contains structural information through headings such as **`<h1>`**, **`<h2>`**, and **`<h3>`**.

In [17]:
html_string = """
<html>
    <body>
        <h1>Artificial Intelligence</h1>
        <p>AI enables machines to perform intelligent tasks.</p>

        <h2>Machine Learning</h2>
        <p>Machine learning allows systems to learn from data.</p>

        <h3>Supervised Learning</h3>
        <p>Models learn from labeled training data.</p>
    </body>
</html>
"""

In [18]:
from langchain_text_splitters import HTMLHeaderTextSplitter

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on)

html_header_splits = html_splitter.split_text(html_string)

html_header_splits

[Document(metadata={'Header 1': 'Artificial Intelligence'}, page_content='Artificial Intelligence'),
 Document(metadata={'Header 1': 'Artificial Intelligence'}, page_content='AI enables machines to perform intelligent tasks.'),
 Document(metadata={'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning'}, page_content='Machine Learning'),
 Document(metadata={'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning'}, page_content='Machine learning allows systems to learn from data.'),
 Document(metadata={'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Supervised Learning'}, page_content='Supervised Learning'),
 Document(metadata={'Header 1': 'Artificial Intelligence', 'Header 2': 'Machine Learning', 'Header 3': 'Supervised Learning'}, page_content='Models learn from labeled training data.')]

### **Conceptual Structure**

```text
<h1> Main Topic
    ↓
<h2> Subtopic
    ↓
<h3> Detail
    ↓
Associated Content
```

The splitter can preserve header information in the resulting document structure/metadata.

> **Key Idea:** When a document has strong structural boundaries, a structure-aware splitter can preserve more useful context than blindly splitting every N characters.

---

# 🧠 **Document-Specific Splitter Mental Model**

```text
Document Type
      ↓
Understand Structure
      ↓
Choose Specialized Splitter
      ↓
Structure-Aware Chunks
```

---

# 4️⃣ **Semantic Splitter / Semantic Chunker**

> **Semantic Chunking = Grouping sentences into chunks based on semantic similarity rather than only fixed character or token size.**

**A semantic chunking approach can be understood as:**

```text
Document
   ↓
Split into sentences
   ↓
Create embeddings for sentences
   ↓
Measure semantic similarity
   ↓
Detect meaningful changes
   ↓
Group related sentences
   ↓
Semantic Chunks
```

### **Core Idea**

**Suppose a document contains:**

```text
Sentence 1 → Transformer architecture
Sentence 2 → Self-attention
Sentence 3 → Multi-head attention
Sentence 4 → Cooking recipe
Sentence 5 → Baking temperature
```

A semantic splitter aims to keep semantically related sentences together and detect stronger topic transitions.

---

# 🧪 **SemanticChunker**

In [19]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

embedding_model = OpenAIEmbeddings()

text_splitter = SemanticChunker(
    embeddings=embedding_model
)

/tmp/ipykernel_3975/231321884.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


### **Split Documents**

In [20]:
chunks = text_splitter.split_documents(docs)

**You can also split raw text:**

In [21]:
chunks = text_splitter.split_text(text)

### **Why Embeddings are Used Here**

Semantic chunking needs a way to estimate whether neighboring sentences are discussing similar concepts.

```text
Sentence A
   ↓
Embedding A

Sentence B
   ↓
Embedding B

Embedding Similarity
   ↓
Semantic Relationship
```

> **Trade-off:** Semantic chunking can produce more meaningful boundaries, but embedding every sentence adds computation and cost compared with a simple character-based splitter.

---

# 🆚 **Text Splitter Comparison**

| Splitter | Main Idea | Best Fit |
|---|---|---|
| **CharacterTextSplitter** | Split using a chosen separator | Simple text splitting |
| **RecursiveCharacterTextSplitter** | Try multiple separators recursively | General-purpose text |
| **Token-based Splitter** | Control chunk size using tokens | Model context-aware chunking |
| **Document-Specific Splitter** | Use document structure | HTML, Markdown, code, etc. |
| **SemanticChunker** | Use semantic similarity between sentences | Meaning-aware chunking |

---

# 🎯 **How to Choose a Text Splitter**

```text
Start
  ↓
Is the document simple prose?
  │
  ├── YES → RecursiveCharacterTextSplitter
  │
  └── NO
       ↓
Does the document have strong structure?
       │
       ├── YES → Document-Specific Splitter
       │
       └── NO
            ↓
Need model-token-aware limits?
            │
            ├── YES → Token-based Splitter
            │
            └── NO
                 ↓
Need semantic boundaries?
                 │
                 └── YES → SemanticChunker
```

---

# 📏 **Chunk Size and Chunk Overlap**

**Two important chunking parameters are:**

```text
chunk_size
→ Target / maximum chunk length according to the splitter's length function

chunk_overlap
→ Amount of overlapping content between neighboring chunks
```

### **Example**

```text
Document
────────────────────────────────────────────

Chunk 1
[ A A A A A A A A ]

Chunk 2
            [ A A A A A A A A ]
            ↑ overlap

Chunk 3
                        [ A A A A A A A A ]
```

### **Why Use Overlap?**

Without overlap, an important sentence can fall at a chunk boundary.

```text
Chunk 1 → "The Transformer uses self-attention..."
Chunk 2 → "...to model relationships between tokens."
```

Overlap gives neighboring chunks some shared context.

> **Important:** More overlap is not automatically better. Excessive overlap increases the number of stored and retrieved tokens.

---

# 💰 **Chunking and RAG Cost**

Chunking decisions affect the rest of the pipeline.

```text
Chunk Size
   ↓
Number of Chunks
   ↓
Number of Embeddings
   ↓
Vector DB Size
   ↓
Retrieval Behavior
   ↓
Prompt Size
   ↓
LLM Cost
```

Therefore, chunking should be evaluated together with retrieval quality and cost.

---

# 🧠 **Embedding + Chunking Relationship**

These two concepts work together.

```text
                 DOCUMENT
                    ↓
                 CHUNKING
                    ↓
          ┌─────────┼─────────┐
          ↓         ↓         ↓
       Chunk 1   Chunk 2   Chunk 3
          ↓         ↓         ↓
      Embedding Embedding Embedding
          ↓         ↓         ↓
          └─────────┼─────────┘
                    ↓
               Vector DB
```

**At query time:**

```text
User Query
    ↓
Query Embedding
    ↓
Vector Similarity
    ↓
Relevant Chunks
```

> **Key Idea:** Chunking controls what gets embedded, while the embedding model controls how that chunk is represented numerically.

---

# ⚠️ **Common Mistakes**

### **1. Using a Random Chunk Size**

There is no universal best chunk size.

```text
Small Chunks
→ Precise retrieval
→ Less surrounding context

Large Chunks
→ More context
→ Potentially more noise
```

Evaluate chunking on your actual documents and queries.

### **2. Confusing Characters with Tokens**

```text
100 characters
≠
100 tokens
```

The relationship depends on the language and tokenizer.

### **3. Ignoring Document Structure**

A structured HTML or code document may benefit from a structure-aware splitter.

### **4. Using Semantic Chunking Everywhere**

Semantic chunking can be powerful, but it requires additional embedding computation.

### **5. Too Much Overlap**

Large overlap can create duplicated context and increase storage and retrieval cost.

---

# 🧪 **Inspect Your Chunks**

Always inspect the actual output of your splitter.

In [22]:
chunks = text_splitter.split_documents(docs)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:5], start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content[:500])

Number of chunks: 30

--- Chunk 1 ---
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or

--- Chunk 2 ---
‡Work performed while at Google Research. 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA, USA.

--- Chunk 3 ---
Recurrent models typically factor computation along the symbol positions of the input and output
sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden
statesht, as a function of the previous hidden stateht−1 and the input for positiont. This inheren

**For debugging RAG systems, inspect:**

```text
Number of chunks
Chunk length
Chunk boundaries
Metadata
Overlap
```

> **Before tuning the retriever, check whether the chunks themselves make sense.**

---

# 🔬 **Quick Revision**

```text
Embedding
→ Text → Numerical Vector

Embedding Model
→ Creates vectors that capture semantic information

Chunking
→ Large Document → Smaller Chunks

CharacterTextSplitter
→ Separator-based splitting

RecursiveCharacterTextSplitter
→ Multiple separators + recursive fallback

Token-based Splitting
→ Control chunking using token count

Document-Specific Splitter
→ Split according to document structure

SemanticChunker
→ Group text using semantic similarity
```

---

# 🧠 **Ultimate Memory Trick**

```text
EMBEDDING
→ What does this text mean?

CHARACTER SPLITTER
→ Split using a separator

RECURSIVE SPLITTER
→ Try larger boundaries first

TOKEN SPLITTER
→ Think in model tokens

DOCUMENT-SPECIFIC
→ Respect document structure

SEMANTIC CHUNKER
→ Split when meaning changes
```

---

# 🏁 **Final Mental Model**

```text
                    DOCUMENT
                        ↓
                 DOCUMENT LOADER
                        ↓
                  TEXT SPLITTER
                        ↓
                     CHUNKS
                        ↓
                 EMBEDDING MODEL
                        ↓
                  VECTOR DATABASE
                        ↓
                    RETRIEVER
                        ↓
                RELEVANT CONTEXT
                        ↓
                       LLM
                        ↓
                     ANSWER
```

### **Interview-Friendly Explanation**

> **Embedding models convert text into numerical vectors so that semantic similarity can be measured. Before creating embeddings, large documents are usually split into smaller chunks. For simple text, CharacterTextSplitter or RecursiveCharacterTextSplitter can be used. Token-aware splitting is useful when model token limits matter, document-specific splitters are useful when structure is important, and SemanticChunker is useful when we want chunk boundaries to follow changes in meaning.**

---

# ✅ **Key Takeaways**

```text
1. Embeddings convert text into vectors.
2. Embeddings are central to semantic search and RAG.
3. Model selection depends on capability, security, cost, latency, token limits, and other task-specific requirements.
4. Chunking is necessary for handling large documents efficiently.
5. RecursiveCharacterTextSplitter is a strong general-purpose option.
6. Token-aware splitting is useful when context limits are important.
7. Document-specific splitters preserve format-specific structure.
8. SemanticChunker uses embeddings to create meaning-aware chunks.
9. Chunk size and overlap should be evaluated rather than chosen blindly.
10. Always inspect the generated chunks before tuning the rest of the RAG pipeline.
```

---